In [ ]:
print("[cell] 01 run parameters (kapi --env): environment variables of this r", flush=True)
# run parameters (kapi --env): environment variables of this run
import os
os.environ['RFS_MODE'] = 'P'
os.environ['RFS_CFG'] = '{"add": ["f_c5s", "v_rl"]}'
os.environ['RFS_UPLOAD'] = 'ngnnguynchn/flag2027-rfsfxc5rl-art-p'

# FLAG 2027 - NB-163: RFS-01 buoc 2, hoc lai cong thuc RF-092 tren kho nguoi sach cua nhom

Code hoc viet lai theo bao cao RF (khong dung artifact RF). `RFS_MODE=L`: LoRA C1 ImageBind-audio. `RFS_MODE=P`: projector QV-Whisper (3 seed) theo fold + cuoi,
luong t3 / ib / nf / nv / age, bo tron g va ng, artifact nap boi `FinalScorer` cua flag_final_core_092 (khong sua core). Luat doc CB: README `Experiment/RFS-01/`.

In [ ]:
print("[boot] FLAG_163_rfs | python", __import__("sys").version.split()[0], "| start", __import__("time").strftime("%Y-%m-%d %H:%M:%S"), flush=True)  # heartbeat rule: first output, before any import
print("[cell] setup: files + deps", flush=True)
import os, sys, json, gzip, base64, subprocess, shutil, time
from pathlib import Path
MODE = os.environ.get("RFS_MODE", "P").upper(); SMOKE = os.environ.get("FLAG_SMOKE") == "1"
W = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./rfs_work"); W.mkdir(exist_ok=True)
CODE = W / "rfs"
for rel, blob in (("train/assemble.py", "H4sIAAAAAAAC/61ZaW/bSBL9rl/R4H4w6ci0ZXuTjBwFmBweDJDNDDJZIYAgEC2xKXElHuBlOo73t++r6uZly0kwWCMx+6iurqquel3Vtizr0/VfJ2eTqdhsVSp8WYpMiusPv/7mTZ6fC1uKU7HC/7UjnjXDZ8K+2VYXYjAXqUKKYivjrViVsb9XYl1KkQW5V2QyjN30VtiqLtw4/ToW1SV9HXck8JPeFtskFjLPVbTaK6I8OUnKQrz7/RNaxEKKd6a1altrbt1szSQaq6ahp7AbC8UL5E11Kd6N3iQQa81qVjIR9qf3v77713tx9KZM1mJy5EyFnIpArpX3KgvW+zD9lgXhKsBvuVHBa+9Vnu7D4jWkv4UWSagJcxmBIjs/n+BTV7ym0muqwRphc1scsU2OxEzAYBvBPe99vNmH+dYt6sIZjyC996qaeOskSveqUN+q87YNniTjNxYAnQhMDP9iW4qiFLT62CP13XVeOVf6nKbiVZXUWFafr75Vq9end8RmzGzu+2yKrUq0GRoWWK7VbQZGNou+w9FBgS3Zb18mhfATaPVRfrwS2yTFts/gHq4gf5kaDtQeGpJ1zbP1a6+j0DM8ejocdS3LGoVRmmSFkNkmlVmuxiJMRkGWRCKVxXYfroQh+BPdhjguI+gmcxGnY9DFPpr4l/qj0fXFOcTey2jlS/FlCgpX5jLL5K39ZUy9YJ/I4uLcGY1Gvgrg7dlG2cegyJ0pezGEIoXXUtthqk8WhuA4YNMUGVnJp18IlFuxSkI9kcMdtTnJMYtQxWJblsSBI2rjsMq0C4XFTEBam7ZenC0dd52kt7bDs0GSIShDxBJNTqZLLRr9hAHN5OJjEqupWCdxEcalaqdX0gff/0LRMA/COCyUja0cV+739gTug84CNMtmc4d7vDxTRZnFRGFsg/W+nSVJQUYujHkyrMxxCMq36Uh43nGzzT5Z2USFPQgAcErZWARWsQ1VKe4wcw+zkYvd0Yp766rZL4PyZkMcTbNhpvZmwxQbdjvBfTHVlxeq8rrUIdukrqrDvMhtR6h9rthMhrssa6+QAKYcardnPU/qt2qvVoBIhBOZhbDx4gX8ncOrGfnnhQOkESB/sw/jHahXAJkIJx9vSqBehOOnkLoSkSyIchOSthl8YNOe+vVYzMHwbjcViyWf8o5O2crTnahCXyBIBIGVAFRZLkeW7dyPv7+gLMjxIkGwJeqKQ7Rb3PpTTgttC0pZY2GxbtxYWU7nXcRvxoNkzFzMTIdtyWtb0iACZeq7mZK+Bxyx2V2ky+BOR3+X358OoMdyxsIvblM1y4sMflL9DIchWA1ZtLJsupj3c3aeqXYm0+soyRKr/W4s4AnAg4iNYlv6joA9cEIWN7ERDdNB6FFqOT1T0c8X7GsQZFEz75oZbg7a4Q4b3xPwEfOGZP0UCbtzTaEew7XIj5fOYHNo4OaqgGvLcl/YlmfBZ+NoDE9BvKepgjG/HNC8pzTchnyAPkZfciIa4q8Zqysawe+H6s9/uP9PGUcf8fet84jmkHk6ZW+2OJlHu1JqMWTYXEXNnpRzPEVxYMfRE5bgEDxkiJstc4F4HFF0H5VAZnuvYruKIMTk/OWZwxdVLOPBhdUpd72g+LeW4hm8HmBnBxHCfYeIwgxQQc8soMXu/vSuurc0cCDPIUt8DVOzAIpGLuhV4ix5LaLfrEVrKY4FiRVE3c7zhztXzc7zn9y5MjtXvZ3nT+wMg/AkAG64JQac/hVAsNrc57J6CPIGdP1wXSBr280WyzFhJ38p1LmB6MaX0OUQGeUi3ECk8JfCgxt1xR86cFrfom1l0JaDqTrvY+xTwGnSXHI/yqNgvjZP9B7g6HfA83tcHmJpd7Ag7SB0B05jjRV+3gPSAxzviPa+H5cHvZQ8AssQVVZ3F2knXD722kOU7CzL74KZQfAD8N1gN3aKo2UTkpT/QCHbIuNaRmENO9JxDjv9QfGqRpH5jxRpvF7Tklf1iU/vUkOcGmKiWP6fERw7HzIC+8Z3rHAD/xgQWgbkGDSljlQeGvC+0XjXR82/C32Pg7261LHez+denHFCFmJSl2M6STtcnsHjOXHjnMumxO2ZztsowEKfU1291KZ6wBRKaMVb5PdYfMTHedQNdhl+wSF1KL2hEhb2eyQNnVau0pklYNYtVqlsRgbT6t/Iaiz+k25Ag+RvRvwX57B1TumQTenQmMcuDoxdDsaY3SaSaYOKexfDAGfHZI3W2HIW03OdcO7J1xIcKETf4FhVRsqg1ovXiR/GmxnwOTh5afFBt5y411+gT/1O56M0jM2pOOPSbDEhchIJKwC/Dieekck7z7oAojhj1wjj4qVW5B/izee35mhxCbu4Zabi7V9zkXSDp4GScB6V60I515Uy7vixqXvXpS5+RYRHhcducvIaxVxEl5Io4EwXqA5RIW4lC1CmHXTC22KKEpuqmi6UF1zAmKPHmRwP+gjzTMVFvlxML/WijBeZMis11VXsOEsyEx0qu4UA5dCn4sYkpD+rwUUrGWid0GNAJN7+8Vm8/fcfv5P5qS7VwWJfnv3yArXV5BfolhRXzMBXFZQMBeae89y5Q5ZYU32jUP7CZAiPNVV4vC2FZObbiF2/d93VD6KgTKHJ0L/dIvG4prd7Qd/WkbWbb2WqFpMlox6uJh9hOiFow1Zm1mnryXoxHYupqWnrgK5oSAC5hgHHV2rO9+CYNQS/x0TaTwwVGaCVimCrDthRuVm1TX72EZjUkpEATauZ1sf04cCVO+WyvKlpewUR3+j969Y8x+nU1EAEaWrWEChznde+1PTomVrCNpJWfGivQEs/jxEQfeghvX4AswYwzMAReJQp4X93ghp0Ao9Spy6+Ayu7C83lFtIxZoBSZbf2WPIajRYz+lCXsy4g3gPmgxKEfqqnpKj+hhTVUIpKSwHw/ZEUgcfJ5MCYTSoSeJRfDqY4JaGZOpiRl0IJ5JZDq5tbvfI43RzONdd75SEFratDVuGktO9ON5XDW+I0ZzKgODmjJFf3K91vH8bgLLYJY74pmhc699dsU0bAqz+pl/VerHY658UDkhHfk01j1TTWOnPQM1SPmS+Pm9yVaQnQqNHdIBY96uIsfd+TRgTbOjmhqnNnPJrETF0Wk2jwCnRlXtoM4NJbGA+50c4PM9sg7+xzVip6vkSB4SU77mqW1wiiOfPtvx4hZ0WdO6cUfFBwXDE4tlFd5/wAiUc6DKsY/+0613zfN3evDqHWR69lm21iD9N+HFBEZlJNJuM2k6F+esBNl1ZiYUFUq6mtOt4PXdn4MSTm5dxr99Gd1qNbIvLmhobbD7lWQzXnPTXnPTUfROy8p+a8p2Y1VHP+pJrzJ9WsPFSRQxamzgSLs4MMGjDoAMXq6BqFHCOqTu4PmAFRTmajBdTpTEDtLtpbGg70hkh3mrBvaeqqpaAmz3P0txQ6Mzc0uuM0OWFz6U6em3sEA7ms1Fd6OsbFb5k/9yAcj4/pKdKuDiyl1MQggHYhil52Ey6BPVOpaPV0iwoSneLh9uy/FLx38XAdIXzvDwuk/+zE8vRqAE2bZiEBg/7b1dTSj6fmBn5iE7jzvsy3JuxHI2jiebGMlOdxIup5hIaeZ2k41NA4+h9GRZ9QcxsAAA=="), ("train/gender.csv", "FF-REDACTED:organisers-file:meta_file_train_set.csv:560-bytes:sha256=fd7991a3353b9f90b2120b8d79fdf50ebaef6438c2e128be162ffbbf6d29c030"), ("train/rfs_train.py", "..."), ("ref/flag_final_core_092.py", "..."), ("ref/manifest_final.json", "H4sIAAAAAAAC/71X247byBF9D5B/KOhpBitp+sbupgcJYA9sZAEnG3sTB0FgCM2+aGhTpEBSmrEX/vecpih5bGSDBAjyMBqy2F116tSlq3/57W9oMXwaxrhbPKPF21er12LFSkFXv6P5xVr6gd68ekb72K/2ru6p6bbUdv1uoC7ReB9p33cfoh9joOR8pBs6djX+x10VQ6jb7UBVTF0f6bWYNrqmHtxYd+2S2oTl7XFJbqB0GGJPw9hHtxuun9Gbd6u/3dcD7NIoAeLHndvGF3Ubbl53b5+v7rinuoIcOvBzpKsfCAuobmFks41twMZUN3G4vqWnr/T6LXUtXY1yCQUZA/6O18vvt9Hd89Wrv/788hYfaPDZge/Rd+S7+64fp8cxDiPg49sw1n5YLCdyT+5stuD3HxDQYpTTF1rU1fzQpvPDcYH/77/Z2P6nO08PYOCrjlFuplBs4uPYu7OeX/IPLaqm8x9z1B/uj7NiWoQ6JwIXls2CXRdik1d1+9i6+ubhFJFV4/ptXB0lxdZjSb+kXXQtdcccLeRE6t0uXjIk1T24kYyGJb50n2ObQdKXC9I5hboepiZ8M5KCi5Nfoe/2mwQJW9snkuMkKU6Spp/eGGOz4CGcBfwkQKLvB8gkRPMS1+8O++zzWQLvhq4dnhof3eGkaNY7xBiGM5100k2ntSTz73v8fJkcgxuxHeqTwmm9YqWelvJSXPy/T5s+HuvzwhMFv0J5jgbTSUiZItPRcCaFcN6VXHpeeSVU6VgwPPnIz+nhxti7h5tjPa6QISvfuGGoUx0zYwtVFc57qYpQ6lJ4mZJKeBEFKyR3KUgng2PxnGvuEGLsUdc3D+4ojtGLGVrfVYdhXAm1Siczp2rKJioXhJEmFaLk2lZeBpl8KE0MyVWuLKz0JkgdZhOz576p96uM+aR/70Z/z1XWJ0UVtBLWWqaC1jGmwLyvouCFc/gQkuHeK764BCIXxpnYKam/BvDf0EPxcX9qbbm1XOX+dn1J4blE/ls+JlWPOeHoairP62+K4v0csm7MgBdPmtLUH7u2+XRLFUpqzLDQyHKBHRVdvWy3aE3314Rar1vgob57GL4yMHSHHr3A9WMNP8bNzrV1Qs+60IIusW73n7NVk7QzPlbIAc+RW1VARrAyMO2sLcB0aUobKimNiMEZWWhfJRGMKEVMJT/n3QR5xr9perX+0FUNuhcsWBWViQopW0qkGNcVc1brElmQVIzIQhOF4dFp7YR1OgWrXYiWSasY9+obC8OD22/ePNWuk0dKGM6rCJRamFKoonQK+EqppEiQWJOM80xqzg3zKqbkC6cZUNhZe8M3E5lnWsCJL2Lk0kiNtE2VTyYpUVXeCKG5rQQ3JhgmiwJJLSuu8ciSNfAv8VSc1Xa929zx9T5zv4joVxzlFlEPupTMCR0EADpbqcI6sGyDtsGzytkiJC2C9FrZwgmhIjd+VnppoesP6F9ZsUiImNKu1IFjU4Euoa012ju4GgtUouasjIVFKWpTeNRktLJAGVqukkjfKwbDJ8TGB88NIGrGiwJYUhmEEkk6hyyIOtqqMJGhGI3hqtAc7EvhhPeJiXRB3B/aje9aMOzHC2jO4ZpEuqGSmZCuQG9QTgVfSWbR0UrlHPNaF1alAsYs+kcsTSVTaQtEYNY9hS1rx4HcjsMlgMhYI3xE2iZnjPQSCeeSLCuFwOoqSleWxrlShdJpWcRgkk3MFE5a5kv9tZqmEeFSOqluXbPZO5x6F1s8OCuN46r0YEuhTry1DnRhwLKFcSDKs1BaB+qFCFUpWBAgk8GfSlj5f0mW/3VMh0O1q4d8juFAS7HHeBDXn+v9lI+5tlAGQUYUBzMGr4gqQ5FHhVaueYgom4rHqFG/JUdJ+WSZFgVSIlVfyQ8RLbqP+XhfvHxE8wb7tO+65hm96x7vYoPORQWBH4yt3b5B6z60XUq1r7HwD69oV/d919Of+26LkN11u/0BWm6O3aPPm9GR//j83WpSREd+cxQk1KxqmU28aOr2IxkyXJ4t3BAvSAlN+dQafsUg5qHz7iXd3dGLv6/+dLf6+fmS+jhE1/t7QkPL4+QIScAk2dcVoIXrNf0F8yC9iO0WEyj1h+zUaVahJh5jcz35/hp9/+A4M5iJ5+l5NWBmbZ68X2Y3bMXO/tM0mdEDRuvu4ZaK8gmJs3Opcdvt6bjpcN70dKxD7ODl+ZZQXAPvDuNfyGfZjiose3o5gO85PuTaMInzCE/BjY6upkdM88vLlcNgHCsxhYqZy5scS1HOYEDFJQJ99PU+kr/vhtiSS+M8fs6aNIJWN+Dyp59e3V6EmvY434G0Hgdo2IJlJGqgLcYAumpxJOIFO2jF1gqXBHPD2ZL2JSPJ16Wk1e/zg1bXt08uSfVADreB3b6pHXIeeqcWlCMOi6eIOmrjAw1g1ucLxJouFy7cdl7h9oSe8q9RjQ/5RjUf9tys5sBnTFdQ/yxDFWpJFlBvCXhOkhk8gJ6M5jlm4ijbuOuCQzoh5fx99B/pitu15nQcCA9lgRo4l9kalffln1quD3wsDgAA")):
    p = CODE / rel; p.parent.mkdir(parents=True, exist_ok=True); p.write_bytes(gzip.decompress(base64.b64decode(blob)))
TRAIN = CODE / "train" / "rfs_train.py"
def run(args, env=None):
    """chay rfs_train.py, chuyen stdout tung dong ra log"""
    p = subprocess.Popen([sys.executable, "-u", str(TRAIN), *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env={**os.environ, **(env or {})})
    for line in p.stdout: print(line, end="", flush=True)
    assert p.wait() == 0, "rfs_train.py failed"
if MODE == "L" and not SMOKE:
    for c in ("pip -q install iopath ftfy regex fvcore", "pip -q install --no-deps git+https://github.com/facebookresearch/ImageBind"):
        print(subprocess.run(c, shell=True, capture_output=True, text=True).stdout[-300:], flush=True)
import torch
print("mode", MODE, "smoke", SMOKE, "cuda", torch.cuda.is_available(), "gpus", torch.cuda.device_count(), flush=True)

In [ ]:
print("[cell] inputs: bundle, lid, roots", flush=True)
IN = Path("/kaggle/input")
def find_all(name):
    return sorted(p for p in IN.rglob(name)) if IN.exists() else []
WORK = W / "work"; WORK.mkdir(exist_ok=True); ART = W / "rfs_art"; BUNDLE = None; LID = ""
if MODE == "B":      # ghep FLAG_162 (a, b, c) + FLAG_160 (whv3 a, b, c) thanh bundle (CPU, vai phut)
    def dsdir(slug_part):
        r = [p for p in IN.rglob("*") if p.is_dir() and p.name.startswith("flag2027-feats-") and slug_part in p.name]
        assert r, f"thieu dataset *{slug_part}*"; return r[0]
    Q = lambda n: str(find_all(n)[0].parent)
    BUNDLE = W / "rfs_bundle"
    cmd = [str(CODE / "train" / "assemble.py")]
    p = subprocess.Popen([sys.executable, "-u", *cmd, "--out", str(BUNDLE), "--rfs_a", str(dsdir("rfs-a")), "--rfs_b", str(dsdir("rfs-b")), "--rfs_c", str(dsdir("rfs-c")),
                          "--wh_a", str(dsdir("whv3-a")), "--wh_b", str(dsdir("whv3-b")), "--wh_c", str(dsdir("whv3-c")), "--extmeta", Q("ext_v1_complete_face_meta.csv"),
                          "--rawv4", Q("train_English.txt"), "--gender_csv", str(CODE / "train" / "gender.csv")], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout: print(line, end="", flush=True)
    assert p.wait() == 0, "assemble failed"
elif SMOKE:
    BUNDLE = W / "bundle_synth"; run(["synth", "--bundle", str(BUNDLE)]); LID = str(BUNDLE / "lid_strict.json")
    if MODE == "LAB":
        TABDIR = W / "lab_tables_synth"; run(["synthlab", "--bundle", str(BUNDLE), "--work", str(TABDIR)])
    torch.save({"params": [torch.zeros(2)]}, WORK / "lora_C1.pt")          # smoke: lora gia (luot L that chay tren Kaggle GPU)
elif True:
    cand = [p.parent for p in find_all("ext.npz") if (p.parent / "v4.npz").exists()]
    assert cand, "thieu bundle (ext.npz + v4.npz): gan dataset dac trung RFS-01 (FLAG_162 + adapter, README 'Hop dong bundle')"
    BUNDLE = cand[0]; l = sorted(find_all("lid_strict.json"), key=lambda q: "lid-strict" not in str(q));   # 13/11: the lid-strict dataset first (other inputs may hold a copy)
    assert l, "thieu lid_strict.json (ngnnguynchn/flag2027-lid-strict-p)"; LID = str(l[0])
    if MODE == "LAB":   # RFS-LAB-01: SAFE-04 pair tables of F1s (ngnnguynchn/flag2027-ct02s-r2ca-tables-p)
        t = find_all(os.environ.get("RFS_LAB_GLOB", "ct01r2_fold*.parquet").replace("*", "1")); assert t, "thieu bang fold SAFE-04 (flag2027-ct02s-r2ca-tables-p)"; TABDIR = t[0].parent
    if MODE in ("P", "LAB"):
        for n in ("lora_C1.pt", "v4_extra.npz"):
            f = find_all(n); assert f, f"thieu {n}: gan dataset dau ra cua luot L"; shutil.copy(f[0], WORK / n)
print("bundle", BUNDLE, "| lid", LID, flush=True)

In [ ]:
print("[cell] lora C1 (ImageBind-audio) + v4 lora/tuoi", flush=True)
if MODE == "L":
    import torch as _t
    ck = W / "imagebind_huge.pth"
    if not ck.exists():
        _t.hub.download_url_to_file("https://huggingface.co/PKU-Alignment/AnyRewardModel/resolve/40c7e2ade237848a194b2b77517da024c4ec8532/imagebind/imagebind_huge.pth", str(ck) + ".part", progress=False); os.replace(str(ck) + ".part", ck)
    tars = [p for p in find_all("tar0000.npz") if p.parent.name == "vox"]; assert tars, "thieu audio cache (duongphihoang/flag2027-audio-cache-p)"
    mav = None
    for d, dirs, files in os.walk(IN):
        if Path(d).name in ("v1", "v2") and {"faces", "voices"} <= set(dirs): mav = Path(d).parent; break
        dirs[:] = [x for x in dirs if x not in ("faces", "voices")] if len(Path(d).relative_to(IN).parts) < 7 else []
    assert mav, "thieu MAV raw (chnnhnnguyn/external-data-mavceleb)"
    v4 = None
    for d, dirs, files in os.walk(IN):
        if "train_English.txt" in files and Path(d, "faces").is_dir(): v4 = Path(d); break
        dirs[:] = [x for x in dirs if x not in ("faces", "voices")] if len(Path(d).relative_to(IN).parts) < 7 else []
    assert v4, "thieu raw-v4 (ngnnguynchn/flag2027-raw-v4-p)"
    run(["lora", "--bundle", str(BUNDLE), "--work", str(WORK), "--lid", LID, "--ib_ckpt", str(ck), "--cache", str(tars[0].parent.parent), "--mav", str(mav), "--v4root", str(v4)])
    OUTD = W / "rfs_lora"; OUTD.mkdir(exist_ok=True)
    for n in ("lora_C1.pt", "v4_extra.npz"): shutil.copy(WORK / n, OUTD / n)
else:
    print("mode P: lora tu dataset cua luot L", flush=True)

In [ ]:
print("[cell] split + oof + proj + fuse + release + selfcheck", flush=True)
if MODE == "P":
    t0 = time.time()
    run(["all", "--bundle", str(BUNDLE), "--work", str(WORK), "--art", str(ART), "--lid", LID, "--lora", str(WORK / "lora_C1.pt")])
    print(f"P done [{time.time() - t0:.0f}s]; artifact:", sorted(p.name for p in ART.iterdir()), flush=True)
    print((ART / "fuse_report.json").read_text(), flush=True)
if MODE == "LAB":   # RFS-LAB-01 (Experiment/EDA-3FAM-01): RFS streams on the SAFE-04 folds, purge per fold
    t0 = time.time()
    run(["lab", "--bundle", str(BUNDLE), "--work", str(WORK), "--art", str(ART), "--lid", LID], env={"RFS_LAB_TABLES": str(TABDIR)})
    print(f"LAB done [{time.time() - t0:.0f}s]:", sorted(p.name for p in ART.iterdir()), flush=True)

In [ ]:
print("[cell] upload", flush=True)
UPL = os.environ.get("RFS_UPLOAD", "")
OUT = {"L": W / "rfs_lora", "B": W / "rfs_bundle"}.get(MODE, ART)
if SMOKE:
    print("smoke: no upload; artifact", sorted(p.name for p in OUT.iterdir()) if OUT.exists() else None, flush=True)
elif UPL:
    try:
        import kagglehub
        kagglehub.dataset_upload(UPL, str(OUT), version_notes=f"RFS-01 FLAG_163 mode {MODE}")
        print("uploaded ->", UPL, flush=True)
    except Exception as e:
        print(f"upload to {UPL} failed ({e!r}); files are in the notebook output", flush=True)
print("ALL CELLS OK", flush=True)